In [ ]:
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from typing import Any

In [ ]:
DB_NAME = "vector_db"
MODEL_NAME = "gemma4"
BASE_URL = "http://localhost:11434"

In [ ]:
### Connect to Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)


In [ ]:
### Setup LangChain objects: llm and retriever

llm = ChatOllama(model=MODEL_NAME, temperature=0, base_url=BASE_URL)
retriever = vector_store.as_retriever()

In [ ]:
### call the implemented invoke() method

retriever.invoke("Who is Avery?")

In [ ]:
llm.invoke("Who is Avery?")

In [ ]:
### Use the retriever and LLM together for RAG

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so explicitly.
Context:
{context}"""

In [ ]:
def answer_question(question: str, history: list[dict[str, str]]) -> str:
    context = retriever.invoke(question)
    context += "\n\n".join(doc.page_content for doc in context)
    sys_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is avery?", [])

In [ ]:
gr.ChatInterface(fn=answer_question).launch()